In [3]:
# %% [markdown]
# ## Cell 1 — Clone + Install (MINIMAL — no imports, no numpy ABI risk)

# %%
import os
import subprocess
import sys
import glob

REPO_URL = "https://github.com/romin4444/multimodal-financial-crisis-prediction.git"
REPO_DIR = "/kaggle/working/fcps"

print("\n" + "="*70)
print("CELL 1: Clone repo + install FCPS")
print("="*70)

# Step 1: Clone or pull
if os.path.isdir(REPO_DIR):
    print(f"[1] Repo exists — pulling...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], 
                   capture_output=True, text=True)
else:
    print(f"[1] Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

# Step 2: Get current HEAD
head = subprocess.run(["git", "log", "-1", "--oneline"], 
                      capture_output=True, text=True).stdout.strip()
print(f"    HEAD: {head}")

# Step 3: Install FCPS core
print("[2] Installing FCPS core (--no-deps)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], 
               check=True)
print("    ✓ Done")

# Step 4: Try to install FinBERT (but don't fail if it doesn't work)
print("[3] Installing FinBERT (optional)...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", 
                   "transformers>=4.30", "datasets>=2.14", "--no-deps"],
                   capture_output=True)
if r.returncode == 0:
    print("    ✓ FinBERT installed")
else:
    print("    ~ FinBERT skipped (scripts will still run)")

# Step 5: List scripts
scripts = sorted(os.path.basename(p) for p in glob.glob("scripts/*.py"))
print(f"\n[OK] {len(scripts)} scripts ready.")
print("    Proceed to Cell 2.\n")


CELL 1: Clone repo + install FCPS
[1] Repo exists — pulling...
    HEAD: 303252c v3.3.0: hazard calibration + vintage FRED scaffold + v4 roadmap
[2] Installing FCPS core (--no-deps)...
    ✓ Done
[3] Installing FinBERT (optional)...
    ✓ FinBERT installed

[OK] 14 scripts ready.
    Proceed to Cell 2.



In [6]:
# ── Secrets + inputs → environment (your fred.py / news.py read these) ────────
import os, glob

# FRED key from Kaggle Secrets → env (fred.py loads os.environ["FRED_API_KEY"])
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_SECRET_FRED_API_KEY"] = UserSecretsClient().get_secret("KAGGLE_SECRET_FRED_API_KEY").strip()
    print("[OK] FRED_API_KEY loaded")
except Exception as e:
    print("[!] No FRED_API_KEY secret — credit/yield/funding + vintage-FSI will be limited:", str(e)[:80])

# Point NEWS_DATA_DIR at the first attached news dataset (load_news auto-detects columns)
news_dirs = sorted(glob.glob("/kaggle/input/datasets")) or sorted(glob.glob("/kaggle/input/datasets"))
if news_dirs:
    os.environ["NEWS_DATA_DIR"] = news_dirs[0]
    print("[OK] NEWS_DATA_DIR =", news_dirs[0])
else:
    print("[!] No news dataset attached — FinBERT ablation will skip cleanly")

# IMPORTANT: the repo ships a 2023-24-only fred_data.csv cache. Delete it so your
# vintage/ALFRED fetch pulls full 1990+ history instead of reusing the short cache.
for c in ["fred_data.csv", "data/fred_data.csv", "data/cache/fred_data.csv"]:
    if os.path.exists(c):
        os.remove(c); print("[removed stale cache]", c)

[OK] FRED_API_KEY loaded
[OK] NEWS_DATA_DIR = /kaggle/input/datasets


In [7]:
import os, subprocess, sys, glob, shutil

os.chdir("/kaggle/working/fcps")
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

SCRIPTS = [
    ("v3 crisis",    "scripts/v3_run.py"),
    ("hazard",       "scripts/hazard_run.py"),
    ("direction",    "scripts/direction_run.py"),
    ("overlay",      "scripts/risk_overlay_run.py"),
    ("v3_advanced",  "scripts/v3_advanced_run.py"),
    ("real_data",    "scripts/real_data_run.py"),
]

for label, script in SCRIPTS:
    if not os.path.exists(script):
        print(f"[SKIP] {script}"); continue
    print(f"\n{'='*60}\n[RUN] {label}\n{'='*60}", flush=True)
    r = subprocess.run([sys.executable, script], capture_output=True, text=True, timeout=1800)
    print((r.stdout + r.stderr)[-3000:])
    print("✓" if r.returncode == 0 else f"✗ code {r.returncode}")

for f in glob.glob("outputs/*.json") + glob.glob("outputs/*.png"):
    shutil.copy(f, "/kaggle/working/artifacts/")

print(f"\n[DONE] {len(os.listdir('/kaggle/working/artifacts'))} artifacts collected")


[RUN] v3 crisis
  FCPS v3 — HONEST WALK-FORWARD BACKTEST (real S&P 500 + VIX)

[1] Loading real market data + engineering features...
    8,815 trading days  1990-01-03 -> 2024-12-30

[2] Building EXOGENOUS label: forward 21d drawdown <= -10%
    {'n': 8814, 'n_positive': 346, 'base_rate': 0.0393}

[3] Computing CAUSAL, online (per-fold refit) regime posteriors...
    Online regime mix: stable=33.7% volatile=37.4% crisis=28.9%

[4] Walk-forward (expanding window, quarterly refit, 21d embargo)...
    BASELINE base-rate           folds=119  PR-AUC=0.0363  BSS=-0.0107  lift@10%=0.1786
    BASELINE VIX-threshold       folds=119  PR-AUC=0.1692  BSS=-7.2168  lift@10%=3.8993
    BASELINE persistence         folds=119  PR-AUC=0.1787  BSS=-6.5363  lift@10%=4.3756
    MODEL price-only (LR, balanced) folds=119  PR-AUC=0.157  BSS=-3.3396  lift@10%=3.9291
    MODEL price-only (LR)        folds=119  PR-AUC=0.1656  BSS=0.0153  lift@10%=3.5719
    MODEL +regime (LR)           folds=119  PR-AUC=0.1282

In [8]:
# ── Read back the key metrics ────────────────────────────────────────────────
import json, glob, os

for name in ["v3_metrics.json", "hazard_metrics.json", "v3_advanced_metrics.json",
             "risk_overlay_results.json", "direction_metrics.json", "metrics_summary.json"]:
    p = os.path.join("outputs", name)
    if os.path.exists(p):
        print(f"\n===== {name} =====")
        print(json.dumps(json.load(open(p)), indent=2)[:2500])

# Sanity check on the calibration fix: hazard Brier skill should no longer be ~ -2
try:
    hz = json.load(open("outputs/hazard_metrics.json"))
    print("\n>>> hazard C-index & Brier skill:", hz)
except Exception:
    pass


===== v3_metrics.json =====
{
  "config": {
    "horizon_days": 21,
    "drawdown_threshold": 0.1,
    "train_head_frac": 0.5,
    "walkforward": {
      "min_train": 1260,
      "step": 63,
      "embargo": 21,
      "horizon": 21
    }
  },
  "label_summary": {
    "n": 8814,
    "n_positive": 346,
    "base_rate": 0.0393
  },
  "metrics": {
    "BASELINE base-rate": {
      "n": 7471,
      "base_rate": 0.045,
      "pr_auc": 0.0363,
      "roc_auc": 0.398,
      "brier": 0.0434,
      "brier_skill": -0.0107,
      "ece": 0.0118,
      "lift_top_decile": 0.1786
    },
    "BASELINE VIX-threshold": {
      "n": 7471,
      "base_rate": 0.045,
      "pr_auc": 0.1692,
      "roc_auc": 0.7503,
      "brier": 0.3529,
      "brier_skill": -7.2168,
      "ece": 0.4875,
      "lift_top_decile": 3.8993
    },
    "BASELINE persistence": {
      "n": 7471,
      "base_rate": 0.045,
      "pr_auc": 0.1787,
      "roc_auc": 0.7936,
      "brier": 0.3237,
      "brier_skill": -6.5363,
      "ec